# Split-MNIST: Continual Learning Benchmark

This notebook compares a **Standard Baseline Neural Network** against our **Dynamic Sprout & Freeze Brain** on the rigorous Split-MNIST Continual Learning benchmark.

### The Experiment
The networks will be trained on 5 tasks sequentially (Digits 0/1, 2/3, 4/5, 6/7, 8/9). Once they finish a task, they are NEVER allowed to see that data again. 

### The Competitors
1. **Baseline MLP:** A standard multi-layer perceptron (784 -> 400 -> 400 -> 10). This is the standard architecture used in Continual Learning research papers (like EWC) to demonstrate Catastrophic Forgetting.
2. **Dynamic Brain:** Our custom architecture equipped with the MAS Pain Receptor. It detects when its capacity is overwhelmed, sprouts new neurons dynamically, and mathematically freezes its old synapses to protect its memories.

## 1. Environment & Data Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1)) # Flatten to 784D
])

print("Downloading MNIST...")
trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

def get_task_data(dataset, class_a, class_b, max_samples=3000):
    idx = (dataset.targets == class_a) | (dataset.targets == class_b)
    X = dataset.data[idx].float() / 255.0
    X = X.view(X.size(0), -1)
    y = dataset.targets[idx]
    perm = torch.randperm(X.size(0))[:max_samples]
    return X[perm], y[perm]

tasks_train = [
    get_task_data(trainset, 0, 1),
    get_task_data(trainset, 2, 3),
    get_task_data(trainset, 4, 5),
    get_task_data(trainset, 6, 7),
    get_task_data(trainset, 8, 9),
]

tasks_test = [
    get_task_data(testset, 0, 1, max_samples=1000),
    get_task_data(testset, 2, 3, max_samples=1000),
    get_task_data(testset, 4, 5, max_samples=1000),
    get_task_data(testset, 6, 7, max_samples=1000),
    get_task_data(testset, 8, 9, max_samples=1000),
]
print("Split-MNIST generated (5 sequential tasks).")

## 2. Architectures & Core Logic

In [ ]:
# ---------------------------------------------------------
# BASELINE COMPETITOR: Standard MLP
# ---------------------------------------------------------
class StandardMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 400)
        self.fc2 = nn.Linear(400, 400)
        self.fc3 = nn.Linear(400, 10)
        
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

# ---------------------------------------------------------
# OUR ARCHITECTURE: The Dynamic Sprout & Freeze Brain
# ---------------------------------------------------------
class DynamicBrain(nn.Module):
    def __init__(self, input_dim=784, output_dim=10):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        
        # Initialize with a tiny structural seed (10 neurons)
        self.hidden_W = nn.Parameter(torch.randn(10, input_dim) * 0.1)
        self.hidden_b = nn.Parameter(torch.zeros(10))
        self.out_W = nn.Parameter(torch.randn(output_dim, 10) * 0.1)
        self.out_b = nn.Parameter(torch.zeros(output_dim))
        
        self.frozen_neurons = 0
        self.active_neurons = 10

    def forward(self, x):
        h = F.relu(F.linear(x, self.hidden_W, self.hidden_b))
        return F.linear(h, self.out_W, self.out_b)
        
    def zero_frozen_grads(self):
        # The mathematical freezing mechanism that protects old memories
        if self.frozen_neurons > 0:
            if self.hidden_W.grad is not None:
                self.hidden_W.grad[:self.frozen_neurons, :] = 0
                self.hidden_b.grad[:self.frozen_neurons] = 0
            if self.out_W.grad is not None:
                self.out_W.grad[:, :self.frozen_neurons] = 0

    def grow_and_freeze(self, num_neurons=20):
        # Lock the old brain
        self.frozen_neurons = self.active_neurons
        self.active_neurons += num_neurons
        
        # Sprout new capacity
        with torch.no_grad():
            new_h_W = torch.randn(num_neurons, self.input_dim, device=self.out_b.device) * 0.1
            new_h_b = torch.zeros(num_neurons, device=self.out_b.device)
            new_o_W = torch.randn(self.output_dim, num_neurons, device=self.out_b.device) * 0.1
            
            self.hidden_W = nn.Parameter(torch.cat([self.hidden_W, new_h_W], dim=0))
            self.hidden_b = nn.Parameter(torch.cat([self.hidden_b, new_h_b], dim=0))
            self.out_W = nn.Parameter(torch.cat([self.out_W, new_o_W], dim=1))

# ---------------------------------------------------------
# THE PAIN RECEPTOR: Memory Aware Synapses (MAS)
# ---------------------------------------------------------
class MASSignal:
    def __init__(self, grow_margin=1.5, fast_alpha=0.1, slow_alpha=0.001, burnin=50):
        self.grow_margin = grow_margin
        self.fast_alpha = fast_alpha
        self.slow_alpha = slow_alpha
        self.burnin = burnin
        self.fast_ema = None
        self.slow_ema = None
        self.step = 0

    def reset(self):
        self.fast_ema = None
        self.slow_ema = None
        self.step = 0

    def score(self, network, X):
        network.zero_grad()
        pred = network(X)
        out_mag = pred.pow(2).mean()
        
        # Measures how hard the network has to fight its own weights to learn this batch
        grads = torch.autograd.grad(out_mag, network.hidden_W, retain_graph=True, allow_unused=True)[0]
        if grads is None: return {'mas': 0.0, 'decision': 'HOLD'}
            
        mas_val = torch.norm(grads).item()
        
        if self.fast_ema is None:
            self.fast_ema = mas_val
            self.slow_ema = mas_val
        else:
            self.fast_ema = (1 - self.fast_alpha) * self.fast_ema + self.fast_alpha * mas_val
            self.slow_ema = (1 - self.slow_alpha) * self.slow_ema + self.slow_alpha * mas_val
            
        self.step += 1
        
        if self.step < self.burnin: return {'mas': mas_val, 'decision': 'BURNIN'}
        if self.fast_ema > self.slow_ema * self.grow_margin: return {'mas': mas_val, 'decision': 'GROW'}
        return {'mas': mas_val, 'decision': 'HOLD'}

print("Architectures Defined.")

## 3. Universal Evaluation Logic

In [ ]:
def evaluate(net, tasks_test, current_task_idx):
    net.eval()
    accs = []
    with torch.no_grad():
        for i in range(current_task_idx + 1):
            X_test, y_test = tasks_test[i]
            X_test, y_test = X_test.to(device), y_test.to(device)
            pred = net(X_test)
            
            # Standard Task-Incremental Evaluation (Forces network to choose between active classes)
            valid_classes = [i * 2, i * 2 + 1]
            mask = torch.ones_like(pred, dtype=torch.bool)
            mask[:, valid_classes] = False
            pred[mask] = -float('inf')
            
            acc = (pred.argmax(dim=1) == y_test).float().mean().item()
            accs.append(acc)
    net.train()
    return accs

## 4. Run Experiment: Baseline Standard MLP
*Watch closely as the accuracy of Task 1 plummets when it learns Task 5.*

In [ ]:
baseline_net = StandardMLP().to(device)
opt_base = torch.optim.Adam(baseline_net.parameters(), lr=0.001)
crit = nn.CrossEntropyLoss()

baseline_accuracies = []

print("Training Standard Baseline MLP...")
for task_idx, (X_train, y_train) in enumerate(tasks_train):
    print(f"\n--- Training Task {task_idx+1}/5 (Digits {task_idx*2}/{task_idx*2+1}) ---")
    
    for epoch in range(2): # 2 epochs per task
        for i in range(0, X_train.size(0), 64):
            bx = X_train[i:i+64].to(device)
            by = y_train[i:i+64].to(device)
            
            opt_base.zero_grad()
            pred = baseline_net(bx)
            
            # Standard MLPs compute loss over all outputs (no task masking crutch)
            loss = crit(pred, by)
            loss.backward()
            opt_base.step()
            
    accs = evaluate(baseline_net, tasks_test, task_idx)
    baseline_accuracies.append(accs)
    print(f"End of Task {task_idx+1} Accuracies:")
    for i, a in enumerate(accs):
        print(f"  Task {i+1}: {a*100:.1f}%")

## 5. Run Experiment: Dynamic Brain
*Watch the MAS Signal detect task boundaries, sprout new neurons, and protect old tasks perfectly.*

In [ ]:
dynamic_net = DynamicBrain(input_dim=784, output_dim=10).to(device)
opt_dyn = torch.optim.Adam(dynamic_net.parameters(), lr=0.005)
signal = MASSignal(grow_margin=1.05, burnin=50)

dynamic_accuracies = []
growth_events = []
total_batches = 0

print("Training Dynamic Sprout & Freeze Brain...")
for task_idx, (X_train, y_train) in enumerate(tasks_train):
    print(f"\n--- Training Task {task_idx+1}/5 (Digits {task_idx*2}/{task_idx*2+1}) ---")
    since_grow = 0
    
    for epoch in range(2):
        for i in range(0, X_train.size(0), 64):
            bx = X_train[i:i+64].to(device)
            by = y_train[i:i+64].to(device)
            if bx.size(0) < 16: continue
                
            res = signal.score(dynamic_net, bx)
            
            opt_dyn.zero_grad()
            pred = dynamic_net(bx)
            
            active_classes = [task_idx * 2, task_idx * 2 + 1]
            loss_mask = torch.ones_like(pred, dtype=torch.bool)
            loss_mask[:, active_classes] = False
            pred[loss_mask] = -float('inf')
            
            loss = crit(pred, by)
            loss.backward()
            
            # -----------------------------------------------------
            # THE HOLY GRAIL MECHANISM: Freeze old memories
            # -----------------------------------------------------
            dynamic_net.zero_frozen_grads() 
            opt_dyn.step()
            
            if res['decision'] == 'GROW' and since_grow > 20:
                print(f"Batch {total_batches:04d} | MAS Spiked! Triggering Structural Plasticity...")
                dynamic_net.grow_and_freeze(num_neurons=20)
                
                # Re-initialize optimizer for the newly added structural parameters
                opt_dyn = torch.optim.Adam(dynamic_net.parameters(), lr=0.005)
                signal.reset()
                since_grow = 0
                growth_events.append(total_batches)
                print(f"  -> Network grew to {dynamic_net.active_neurons} neurons. Old knowledge mathematically FROZEN.")
                
            total_batches += 1
            since_grow += 1
            
    accs = evaluate(dynamic_net, tasks_test, task_idx)
    dynamic_accuracies.append(accs)
    print(f"End of Task {task_idx+1} Accuracies:")
    for i, a in enumerate(accs):
        print(f"  Task {i+1}: {a*100:.1f}%")

## 7. Hard Mode: Split-FashionMNIST
MNIST is notoriously easy. Let's see if the architecture survives a significantly harder dataset where catastrophic forgetting usually strikes much faster.

In [ ]:
print("\nDownloading FashionMNIST...")
fashion_train = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
fashion_test = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

fashion_tasks_train = [get_task_data(fashion_train, i*2, i*2+1) for i in range(5)]
fashion_tasks_test = [get_task_data(fashion_test, i*2, i*2+1, max_samples=1000) for i in range(5)]

# --- FASHION BASELINE ---
fashion_baseline_net = StandardMLP().to(device)
opt_base_f = torch.optim.Adam(fashion_baseline_net.parameters(), lr=0.001)
fashion_baseline_accuracies = []

print("Training Standard Baseline MLP on FashionMNIST...")
for task_idx, (X_train, y_train) in enumerate(fashion_tasks_train):
    print(f"\n--- Training Task {task_idx+1}/5 ---")
    for epoch in range(2):
        for i in range(0, X_train.size(0), 64):
            bx = X_train[i:i+64].to(device)
            by = y_train[i:i+64].to(device)
            opt_base_f.zero_grad()
            pred = fashion_baseline_net(bx)
            loss = crit(pred, by)
            loss.backward()
            opt_base_f.step()
            
    accs = evaluate(fashion_baseline_net, fashion_tasks_test, task_idx)
    fashion_baseline_accuracies.append(accs)
    print(f"End of Task {task_idx+1} Accuracies:")
    for i, a in enumerate(accs):
        print(f"  Task {i+1}: {a*100:.1f}%")

# --- FASHION DYNAMIC ---
fashion_net = DynamicBrain(input_dim=784, output_dim=10).to(device)
opt_fashion = torch.optim.Adam(fashion_net.parameters(), lr=0.005)
fashion_signal = MASSignal(grow_margin=1.05, slow_alpha=0.001, burnin=50)
fashion_dynamic_accuracies = []
total_batches_f = 0

print("\nTraining Dynamic Brain on FashionMNIST...")
for task_idx, (X_train, y_train) in enumerate(fashion_tasks_train):
    print(f"\n--- Training Task {task_idx+1}/5 ---")
    since_grow = 0
    for epoch in range(2):
        for i in range(0, X_train.size(0), 64):
            bx = X_train[i:i+64].to(device)
            by = y_train[i:i+64].to(device)
            if bx.size(0) < 16: continue
            
            res = fashion_signal.score(fashion_net, bx)
            opt_fashion.zero_grad()
            pred = fashion_net(bx)
            
            active_classes = [task_idx * 2, task_idx * 2 + 1]
            loss_mask = torch.ones_like(pred, dtype=torch.bool)
            loss_mask[:, active_classes] = False
            pred[loss_mask] = -float('inf')
            
            loss = crit(pred, by)
            loss.backward()
            fashion_net.zero_frozen_grads() 
            opt_fashion.step()
            
            if res['decision'] == 'GROW' and since_grow > 20:
                print(f"  Batch {total_batches_f:04d} | MAS Spiked! Sprouting new capacity...")
                fashion_net.grow_and_freeze(num_neurons=20)
                print(f"  -> Network grew to {fashion_net.active_neurons} neurons. Old knowledge FROZEN.")
                opt_fashion = torch.optim.Adam(fashion_net.parameters(), lr=0.005)
                fashion_signal.reset()
                since_grow = 0
                
            total_batches_f += 1
            since_grow += 1
            
    accs = evaluate(fashion_net, fashion_tasks_test, task_idx)
    fashion_dynamic_accuracies.append(accs)
    print(f"End of Task {task_idx+1} Accuracies:")
    for i, a in enumerate(accs):
        print(f"  Task {i+1}: {a*100:.1f}%")



## 6. Final Results & Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
tasks = [1, 2, 3, 4, 5]
colors = ['b', 'g', 'r', 'c', 'm']

def plot_accuracies(ax, acc_list, title):
    for task_id in range(5):
        task_accs = []
        eval_steps = []
        for i, task_res in enumerate(acc_list):
            if task_id < len(task_res):
                task_accs.append(task_res[task_id] * 100)
                eval_steps.append(i + 1)
        if task_accs:
            ax.plot(eval_steps, task_accs, marker='o', color=colors[task_id], label=f'Task {task_id+1}')
    ax.set_title(title)
    ax.set_xlabel('End of Task Step')
    ax.set_ylabel('Accuracy %')
    ax.set_xticks(tasks)
    ax.set_ylim(0, 105)
    ax.legend()
    ax.grid(True, alpha=0.3)

plot_accuracies(axes[0, 0], baseline_accuracies, 'MNIST Standard Baseline (Catastrophic Forgetting)')
plot_accuracies(axes[0, 1], dynamic_accuracies, 'MNIST Dynamic Sprout & Freeze')
plot_accuracies(axes[1, 0], fashion_baseline_accuracies, 'FashionMNIST Standard Baseline (Catastrophic Forgetting)')
plot_accuracies(axes[1, 1], fashion_dynamic_accuracies, 'FashionMNIST Dynamic Sprout & Freeze')

plt.tight_layout()
plt.show()

